# Short-Form Video Ad Market Analysis

Competitive analysis of the short-form video advertising landscape (**TikTok**, **Instagram Reels**, **YouTube Shorts**) comparing monetization models and ad-load strategies to identify where each platform captures advertiser spend, and where ad budgets are likely to shift.

**Author:** Harry Vo  
**Data note:** Figures are seeded from publicly reported 2023-2025 estimates (earnings calls, eMarketer/Insider Intelligence, press reporting). They are a transparent, reproducible approximation rather than audited financials. See `data/generate_data.py` to edit assumptions.

In [1]:
import os, sys
# Anchor to the repo root so relative paths work whether the kernel starts
# in the repo root or in notebooks/.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('working dir:', os.getcwd())

working dir: /sessions/sweet-busy-meitner/mnt/outputs/short-form-video-ad-analysis


## 1. Setup & data generation

In [2]:
import sys, subprocess
subprocess.run([sys.executable, 'data/generate_data.py'], check=True)
import pandas as pd
plats = pd.read_csv('data/platforms.csv')
proj  = pd.read_csv('data/projections.csv')
plats

Wrote data/platforms.csv and data/projections.csv


,platform,owner,ad_rev_2024_b,mau_m,ad_load_pct,ad_arpu_usd,monetization
0,TikTok,ByteDance,23.6,1100,9.0,21.45,in-feed + Spark + Shop
1,Instagram Reels,Meta,22.0,2000,6.5,11.00,in-feed (Meta Ads)
2,YouTube Shorts,Alphabet,8.5,2000,4.0,4.25,in-feed (skippable) + share


## 2. Competitive metrics

Revenue share, ad ARPU (ad $ per monthly active user / yr), and a **monetization efficiency** ratio (ARPU per point of ad load) — a proxy for how much each platform earns per unit of ad inventory.

In [3]:
from src.analysis import competitive_metrics, headroom
metrics = competitive_metrics(plats)
metrics[['platform','ad_rev_2024_b','rev_share_pct','ad_arpu_usd','ad_load_pct','monetization_efficiency']]

,platform,ad_rev_2024_b,rev_share_pct,ad_arpu_usd,ad_load_pct,monetization_efficiency
0,TikTok,23.6,43.6,21.45,9.0,2.38
1,Instagram Reels,22.0,40.7,11.00,6.5,1.69
2,YouTube Shorts,8.5,15.7,4.25,4.0,1.06


## 3. Ad-load headroom

Distance from each platform's current ad load to the highest-load platform — a rough ceiling on how much more inventory could be introduced before parity.

In [4]:
headroom(plats)

,platform,ad_load_pct,ad_load_headroom_pct
0,TikTok,9.0,0.0
1,Instagram Reels,6.5,2.5
2,YouTube Shorts,4.0,5.0


## 4. Five-year revenue projection

In [5]:
import matplotlib.pyplot as plt
for name, g in proj.groupby('platform'):
    plt.plot(g['year'], g['ad_rev_b'], marker='o', label=name)
plt.title('Projected short-form ad revenue (2024-2028)')
plt.ylabel('Ad revenue ($B)'); plt.xlabel('Year'); plt.legend(); plt.show()

## 5. Recommendation

- **TikTok** leads absolute ad revenue share and pairs the highest ad load with a commerce layer (Shop/Spark), giving it the most diversified take rate.
- **YouTube Shorts** carries the lowest ad load and the most ad-load headroom, so its revenue growth is the steepest in the projection — the clearest place incremental advertiser spend can be unlocked.
- **Reels** benefits from Meta's ad stack and scale but sits mid-pack on monetization efficiency.

**Where budgets shift:** dollars follow inventory that is still under-monetized with strong reach — favoring Shorts for incremental spend, while TikTok defends share through commerce-led formats.

## 6. Full script run

`src/analysis.py` reproduces every metric and saves figures to `outputs/`.

In [6]:
from src import analysis
analysis.main()


=== Competitive metrics (2024) ===
       platform  ad_rev_2024_b  rev_share_pct  ad_arpu_usd  ad_load_pct  monetization_efficiency
         TikTok           23.6           43.6        21.45          9.0                     2.38
Instagram Reels           22.0           40.7        11.00          6.5                     1.69
 YouTube Shorts            8.5           15.7         4.25          4.0                     1.06

=== Ad-load headroom ===
       platform  ad_load_pct  ad_load_headroom_pct
         TikTok          9.0                   0.0
Instagram Reels          6.5                   2.5
 YouTube Shorts          4.0                   5.0

=== Projected 2028 ad revenue ===
       platform  year  ad_rev_b
         TikTok  2028     72.22
Instagram Reels  2028     57.45
 YouTube Shorts  2028     37.70

Saved outputs/fig_revenue_2024.png, outputs/fig_projection.png, and outputs/competitive_metrics.csv

=== Takeaway ===
- TikTok leads 2024 ad revenue share at 43.6%.
- TikTok shows th